### Retiro Fugas

In [3]:

import sys 
sys.path.append('C:/Users/Data/Documents/lazo_fernando/target_script_01/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-09-01'

filename='actualizar_campos_20260902.xlsx'
name_dni='DNI'

ruta_archivo = os.path.join(ruta_efectiva, filename)
df = pd.read_excel(ruta_archivo)

df[f"{name_dni}"] = (
    df[f"{name_dni}"]
    .astype(str)
    .str.zfill(8)
)
print(df.shape)
df = df.rename(columns={
    f'{name_dni}': 'NUMDOCUMENTO'
})
df.head()
server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)



(159484, 2)


In [4]:
df.head()

,NUMDOCUMENTO,AREAEFECTINEGOCIOS
0,40688943,VILLA EL SALVADOR
1,42577725,VILLA EL SALVADOR
2,40680453,VILLA EL SALVADOR
3,09689344,VILLA EL SALVADOR
4,10595116,VILLA EL SALVADOR


In [5]:

df.to_sql(
    name="cruce_efectiva",
    con=engine_kishin,
    if_exists="append",
    index=False,
    chunksize=1000
)


159484

In [6]:
df.shape

(29302, 3)

In [7]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE a
            SET a.AREAEFECTINEGOCIOS = b.AREAEFECTINEGOCIOS
            from DANTALION.dbo.Base_Maestra_Efectiva_Negocios a
            inner join DANTALION.dbo.cruce_efectiva B
            ON A.NUMDOCUMENTO=B.NUMDOCUMENTO
            WHERE A.fecha_envio>='2026-09-01'
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 159484


In [11]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE a
            SET a.CASHBACK=case
                when PERFIL ='PERFIL SE' then 'CASHBACK 450'
                else 'CASHBACK 250'
            end 
            from DANTALION.dbo.Base_Maestra_Diners_TC a
            WHERE fecha_envio>='2026-08-22'
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 29302


In [ ]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE a
            SET a.N_BASE = b.fuente
            ,a.RECURENCIA = b.RECENCIA
            from DANTALION.dbo.Base_Maestra_Diners_TC a
            inner join DANTALION.dbo.cruce_dinerstc B
            ON A.NUMERO_DOCUMENTO=B.NUMERO_DOCUMENTO
            WHERE A.fecha_envio>='2026-08-01'
            and A.fecha_envio<'2026-09-01'
            and b.mes='julio'
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 36911


In [9]:
server_sql = server_zeus
db_sql = "MAEBA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_MAEBA = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)


In [11]:
df.to_sql(
    name="cruce_dinerstc",
    con=engine_MAEBA,
    if_exists="append",
    index=False,
    chunksize=1000
)


15195

In [14]:
try:
    with engine_MAEBA.begin() as conn:
        query = f"""
            UPDATE a
            SET a.N_BASE = b.fuente
            from MAEBA.ADM_OBJ_TG.tGestionMesDinersTc a
            inner join MAEBA.dbo.cruce_dinerstc B
            ON A.NUMERO_DOCUMENTO COLLATE Latin1_General_CI_AI=B.NUMERO_DOCUMENTO
            WHERE A.AÑO_DURACION_BASE=2026
            and  a.tMesGestion=b.mes
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 50145


In [8]:
from sqlalchemy import text

with engine_kishin.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS cruce_efectiva"))
    # conn.execute(text("TRUNCATE TABLE tb_funnel_reclutamiento"))

In [9]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners_tc", "SP tNumeros diners TC")
# exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC Zeus")
# exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC SA")

SP tNumeros diners TC | realizado | duración: 33.55 seg
